# Stock Return Prediction Using Ridge & Lasso Regression
**B.Tech Machine Learning Mini-Project — Reliance Industries (RELIANCE.NS)**

This notebook demonstrates the complete Machine Learning workflow:
1. Loading and validating the raw Reliance Industries stock dataset (`RELIANCE.NS_1973-05-08_2025-03-01.csv`).
2. Cleaning the data and engineering leakage-free features (`daily_return`, `intraday_return`, `daily_range`, `lag_return_1`, `return_5d`).
3. Defining the next-trading-day target (`target_return = close_{t+1} / close_t - 1`).
4. Exploratory Data Analysis (EDA).
5. Chronological train/test split (80% training, 20% unseen testing) and training **Ridge Regression ($L_2$)** and **Lasso Regression ($L_1$)** inside `scikit-learn` Pipelines.
6. Evaluating both models using MAE, MSE, RMSE, $R^2$ Score, and Directional Accuracy.
7. Generating a sample next-trading-day return prediction.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import run_preprocessing, FEATURE_COLUMNS, TARGET_COLUMN
from src.train import train_and_tune_models
from src.evaluate import evaluate_trained_models
from src.predict import get_default_prediction_inputs, predict_next_day_return

## 1. Data Loading, Validation, Cleaning & Feature Engineering

In [ ]:
clean_df, quality_summary = run_preprocessing(save_outputs=True)
print("Data Quality Summary:")
for k, v in quality_summary.items():
    if not isinstance(v, (dict, list)):
        print(f"  {k:40s}: {v}")

clean_df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(clean_df["date"], clean_df["close"], color="#1f4e78")
axes[0].set_title("Reliance Historical Closing Price Trend (INR)")
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Close Price (INR)")

axes[1].hist(clean_df["daily_return"] * 100.0, bins=70, color="#0284c7", edgecolor="white")
axes[1].set_title("Distribution of Daily Returns (%)")
axes[1].set_xlabel("Daily Return (%)")
axes[1].set_ylabel("Frequency")
plt.tight_layout()
plt.show()

## 3. Chronological Split & Model Training (Ridge vs. Lasso)

In [ ]:
training_bundle = train_and_tune_models(clean_df, save_models=True)
eval_bundle = evaluate_trained_models(
    ridge_pipeline=training_bundle["ridge_pipeline"],
    lasso_pipeline=training_bundle["lasso_pipeline"],
    test_df=training_bundle["test_df"],
    save_outputs=True,
)
eval_bundle["metrics_df"]

## 4. Sample Next-Trading-Day Return Prediction

In [ ]:
defaults = get_default_prediction_inputs()
for model_name in ["Lasso Regression", "Ridge Regression"]:
    pred = predict_next_day_return(defaults, model_choice=model_name)
    print(f"{model_name}: {pred['predicted_return_pct']:+.4f}% -> {pred['interpretation']}")